# PlantVillage: тестирование моделей

Локальный тест моделей перед переходом на FL.  
Данные: `data/plantvillage` (54 305 изображений, 38 классов, 256×256).  
Разбивка: 80% train / 20% val (стратифицированная, seed=42).  
Модели: `efficientnet_b0` (pretrained ImageNet, fine-tune 224×224).

In [ ]:
import os
import time
import numpy as np
import torch
import torch.nn as nn
import timm
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from datasets import load_from_disk
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from plantvillage_models import build_model, MODEL_CONFIGS

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
print(f"torch  : {torch.__version__}")
print()
print("Доступные модели:")
for name, cfg in MODEL_CONFIGS.items():
    print(f"  {name:<25s} — {cfg['description']}")

## 1. Конфигурация

In [ ]:
# ── выбор модели ─────────────────────────────────────────────────
MODEL_NAME = "efficientnet_b0"   # "efficientnet_b0" | "efficientnet_b0_scratch" | "mobilenetv3_small"

cfg = MODEL_CONFIGS[MODEL_NAME]

# ── данные ───────────────────────────────────────────────────────
DATA_DIR    = "data/plantvillage"
NUM_CLASSES = 38
IMAGE_SIZE  = cfg["image_size"]
VAL_SPLIT   = 0.2
SPLIT_SEED  = 42

# ── обучение (зависит от режима) ──────────────────────────────────
IS_PRETRAINED = cfg.get("pretrained", True)

if IS_PRETRAINED:
    PHASE1_EPOCHS = cfg["phase1_epochs"]
    PHASE2_EPOCHS = cfg["phase2_epochs"]
    NUM_EPOCHS    = PHASE1_EPOCHS + PHASE2_EPOCHS
else:
    NUM_EPOCHS    = cfg["epochs"]

BATCH_SIZE   = cfg["batch_size"]
LABEL_SMOOTH = cfg["label_smooth"]
MIXUP_ALPHA  = cfg["mixup_alpha"]
NUM_WORKERS  = 4
USE_AMP      = True

# ── сохранение ───────────────────────────────────────────────────
SAVE_DIR = f"runs/plantvillage_local/{MODEL_NAME}"
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"Model        : {MODEL_NAME}")
print(f"Config       : {cfg['description']}")
print(f"Pretrained   : {IS_PRETRAINED}")
print(f"Image size   : {IMAGE_SIZE}×{IMAGE_SIZE}")
if IS_PRETRAINED:
    print(f"Phase 1      : {PHASE1_EPOCHS} эпох  (backbone frozen, lr_head={cfg['lr_head_p1']})")
    print(f"Phase 2      : {PHASE2_EPOCHS} эпох  (BN frozen, lr_backbone={cfg['lr_backbone']}, lr_head={cfg['lr_head_p2']})")
else:
    print(f"Epochs       : {NUM_EPOCHS}  |  LR: {cfg['lr']}  |  WD: {cfg['weight_decay']}")
print(f"Total epochs : {NUM_EPOCHS}")
print(f"Mixup        : {MIXUP_ALPHA}  |  LabelSmooth: {LABEL_SMOOTH}  |  AMP: {USE_AMP}")
print(f"Save dir     : {SAVE_DIR}")

## 2. Датасет и даталоадер

In [ ]:
# ImageNet mean/std — используем для pretrained моделей
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)


class AddGaussianNoise:
    """Случайный гауссов шум — имитирует сенсорный шум дешёвых камер."""
    def __init__(self, std=0.05):
        self.std = std

    def __call__(self, tensor):
        return tensor + torch.randn_like(tensor) * self.std  # без clamp — тензор уже нормализован


# ── Лёгкая аугментация (лабораторные условия) ────────────────────
train_tf_light = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# ── Сильная аугментация (полевые условия) ────────────────────────
# Имитирует реальные условия съёмки: разный угол, освещение, шум, частичное перекрытие
train_tf_heavy = transforms.Compose([
    transforms.Resize((IMAGE_SIZE + 32, IMAGE_SIZE + 32)),
    transforms.RandomCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(degrees=30),
    transforms.RandomPerspective(distortion_scale=0.3, p=0.5),
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5, hue=0.1),
    transforms.RandomGrayscale(p=0.05),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    AddGaussianNoise(std=0.03),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.15), ratio=(0.3, 3.3)),
])

# ── Выбор аугментации ─────────────────────────────────────────────
AUGMENTATION = "heavy"   # "light" | "heavy"
train_tf = train_tf_heavy if AUGMENTATION == "heavy" else train_tf_light
print(f"Augmentation : {AUGMENTATION}")

val_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class PlantVillageDataset(Dataset):
    def __init__(self, hf_split, indices, transform):
        self.data      = hf_split
        self.indices   = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        sample = self.data[int(self.indices[idx])]
        img    = sample["image"].convert("RGB")
        label  = sample["label"]
        return self.transform(img), label


# Загрузка и стратифицированный split
hf_ds      = load_from_disk(DATA_DIR)
hf_train   = hf_ds["train"]
CLASS_NAMES = hf_train.features["label"].names

all_labels  = hf_train["label"]
train_idx, val_idx = train_test_split(
    np.arange(len(hf_train)),
    test_size=VAL_SPLIT,
    stratify=all_labels,
    random_state=SPLIT_SEED,
)

train_ds = PlantVillageDataset(hf_train, train_idx, train_tf)
val_ds   = PlantVillageDataset(hf_train, val_idx,   val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f"Total  : {len(hf_train):,} samples, {NUM_CLASSES} классов")
print(f"Train  : {len(train_ds):,} samples  ({len(train_loader)} batches)")
print(f"Val    : {len(val_ds):,} samples  ({len(val_loader)} batches)")
print(f"Classes: {CLASS_NAMES[:4]} ...")

## 3. Инициализация модели

In [ ]:
model = build_model(MODEL_NAME, num_classes=NUM_CLASSES).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model        : {MODEL_NAME}")
print(f"Total params : {total_params:,}  ({total_params/1e6:.1f}M)")

dummy = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE)
with torch.no_grad():
    out = model(dummy)
print(f"Output shape : {out.shape}  ✓")

## 4. Параметры обучения

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
print("Loss: CrossEntropyLoss(label_smoothing={})".format(LABEL_SMOOTH))
print("Optimizer и scheduler настраиваются в ячейке обучения (2-phase)")

## 5. Обучение

In [ ]:
from copy import deepcopy

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "val_f1": []}
best_val_acc = 0.0
best_ckpt    = os.path.join(SAVE_DIR, "best.pt")
log_path     = os.path.join(SAVE_DIR, "train.log")
scaler       = torch.amp.GradScaler("cuda", enabled=USE_AMP)

N_TRAIN = len(train_loader)
N_VAL   = len(val_loader)


def freeze_bn(model):
    for m in model.modules():
        if isinstance(m, torch.nn.BatchNorm2d):
            m.eval()
            m.weight.requires_grad_(False)
            m.bias.requires_grad_(False)


def mixup_data(x, y, alpha):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam


def mixup_loss(criterion, logits, y_a, y_b, lam):
    return lam * criterion(logits, y_a) + (1 - lam) * criterion(logits, y_b)


header = f"{'Epoch':>5}  {'Phase':>7}  {'T-Loss':>8}  {'T-Acc':>7}  {'V-Loss':>8}  {'V-Acc':>7}  {'V-F1':>7}  {'LR-head':>9}  {'Time':>6}"
sep    = "-" * len(header)


def log(line, file=None):
    print(line)
    if file:
        file.write(line + "\n")
        file.flush()


def train_epoch(optimizer, phase_label, freeze_batchnorm=False):
    model.train()
    if freeze_batchnorm:
        freeze_bn(model)
    total_loss, correct, total = 0.0, 0, 0
    for i, (imgs, labels) in enumerate(train_loader, 1):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            if MIXUP_ALPHA > 0:
                imgs, y_a, y_b, lam = mixup_data(imgs, labels, MIXUP_ALPHA)
                logits = model(imgs)
                loss   = mixup_loss(criterion, logits, y_a, y_b, lam)
                preds  = logits.argmax(1)
                correct += (lam * (preds == y_a).float() + (1 - lam) * (preds == y_b).float()).sum().item()
            else:
                logits = model(imgs)
                loss   = criterion(logits, labels)
                correct += (logits.argmax(1) == labels).sum().item()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * len(labels)
        total      += len(labels)
        print(f"  train {i:>4}/{N_TRAIN}  loss={total_loss/total:.4f}  acc={correct/total*100:.1f}%",
              end="\r", flush=True)
    print(" " * 60, end="\r")
    return total_loss / total, correct / total


def val_epoch():
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels_list = [], []
    with torch.no_grad():
        for i, (imgs, labels) in enumerate(val_loader, 1):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                logits = model(imgs)
                loss   = criterion(logits, labels)
            preds = logits.argmax(1)
            total_loss += loss.item() * len(labels)
            correct    += (preds == labels).sum().item()
            total      += len(labels)
            all_preds.extend(preds.cpu().numpy())
            all_labels_list.extend(labels.cpu().numpy())
            print(f"  val   {i:>4}/{N_VAL}  loss={total_loss/total:.4f}  acc={correct/total*100:.1f}%",
                  end="\r", flush=True)
    print(" " * 60, end="\r")
    from sklearn.metrics import precision_recall_fscore_support
    _, _, f1, _ = precision_recall_fscore_support(
        all_labels_list, all_preds, average="macro", zero_division=0
    )
    return total_loss / total, correct / total, f1


def run_epochs(optimizer, scheduler, epochs_range, phase_label, freeze_batchnorm, log_file):
    global best_val_acc
    for epoch in epochs_range:
        t0 = time.time()
        tr_loss, tr_acc         = train_epoch(optimizer, phase_label, freeze_batchnorm)
        vl_loss, vl_acc, vl_f1 = val_epoch()
        scheduler.step()
        elapsed    = time.time() - t0
        current_lr = scheduler.get_last_lr()[-1]
        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(vl_loss)
        history["val_acc"].append(vl_acc)
        history["val_f1"].append(vl_f1)
        marker = " ★" if vl_acc > best_val_acc else ""
        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            torch.save(model.state_dict(), best_ckpt)
        line = (f"{epoch:>5}  {phase_label:>7}  {tr_loss:>8.4f}  {tr_acc*100:>6.2f}%  "
                f"{vl_loss:>8.4f}  {vl_acc*100:>6.2f}%  {vl_f1:.4f}  "
                f"{current_lr:>9.1e}  {elapsed:>5.0f}s{marker}")
        log(line, log_file)


with open(log_path, "w") as f:
    log(f"Model: {MODEL_NAME}  |  {cfg['description']}", f)
    log("", f)
    log(header, f)
    log(sep, f)

    if IS_PRETRAINED:
        # ── Phase 1: backbone заморожен ──────────────────────────────
        print("=" * 60)
        print(f"PHASE 1: {PHASE1_EPOCHS} эпох  —  backbone frozen, lr_head={cfg['lr_head_p1']}")
        print("=" * 60)
        for name, param in model.named_parameters():
            param.requires_grad_("classifier" in name)
        opt1   = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=cfg["lr_head_p1"], weight_decay=cfg["weight_decay"],
        )
        sched1 = torch.optim.lr_scheduler.CosineAnnealingLR(opt1, T_max=PHASE1_EPOCHS, eta_min=1e-5)
        run_epochs(opt1, sched1, range(1, PHASE1_EPOCHS + 1), "head", freeze_batchnorm=True, log_file=f)

        # ── Phase 2: весь backbone, BN заморожен ─────────────────────
        print()
        print("=" * 60)
        print(f"PHASE 2: {PHASE2_EPOCHS} эпох  —  full model, BN frozen")
        print(f"  lr_backbone={cfg['lr_backbone']}, lr_head={cfg['lr_head_p2']}")
        print("=" * 60)
        for param in model.parameters():
            param.requires_grad_(True)
        backbone_params = [p for n, p in model.named_parameters() if "classifier" not in n]
        head_params     = [p for n, p in model.named_parameters() if "classifier" in n]
        opt2   = torch.optim.AdamW([
            {"params": backbone_params, "lr": cfg["lr_backbone"]},
            {"params": head_params,     "lr": cfg["lr_head_p2"]},
        ], weight_decay=cfg["weight_decay"])
        sched2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=PHASE2_EPOCHS, eta_min=1e-7)
        log(sep, f)
        run_epochs(opt2, sched2, range(PHASE1_EPOCHS + 1, NUM_EPOCHS + 1), "full", freeze_batchnorm=True, log_file=f)

    else:
        # ── Scratch: обычное обучение ─────────────────────────────────
        print("=" * 60)
        print(f"SCRATCH: {NUM_EPOCHS} эпох  —  lr={cfg['lr']}, wd={cfg['weight_decay']}")
        print("=" * 60)
        optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
        run_epochs(optimizer, scheduler, range(1, NUM_EPOCHS + 1), "scratch", freeze_batchnorm=False, log_file=f)

    log("", f)
    log(f"Best val acc: {best_val_acc*100:.2f}%  → {best_ckpt}", f)

print()
print(f"Best val acc: {best_val_acc*100:.2f}%")
print(f"Log saved  → {log_path}")

## 6. Оценка качества (best checkpoint)

In [ ]:
# ── Оценка на чистом val (стандартная) ──────────────────────────
best_ckpt = os.path.join(SAVE_DIR, "best.pt")  # восстанавливаем если ячейка запущена отдельно
model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
model.eval()


def evaluate(loader, desc=""):
    all_preds, all_labels_list = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                preds = model(imgs).argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels_list.extend(labels.numpy())
    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels_list)
    acc = accuracy_score(all_labels, all_preds)
    prec, rec, f1, support = precision_recall_fscore_support(
        all_labels, all_preds, average=None, zero_division=0
    )
    prec_m, rec_m, f1_m, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="macro", zero_division=0
    )
    print(f"\n{'─'*55}")
    print(f"  {desc}")
    print(f"{'─'*55}")
    print(f"  Accuracy  : {acc*100:.2f}%")
    print(f"  F1 macro  : {f1_m:.4f}")
    print(f"  Precision : {prec_m*100:.2f}%  (macro)")
    print(f"  Recall    : {rec_m*100:.2f}%  (macro)")
    worst = np.argsort(f1)[:5]
    print(f"\n  Топ-5 худших классов:")
    for i in worst:
        print(f"    [{i:>2}] {CLASS_NAMES[i]:<45s}  F1={f1[i]:.3f}  n={support[i]}")
    return acc, f1_m, all_preds, all_labels


# Чистый val
acc_clean, f1_clean, preds_clean, labels_clean = evaluate(val_loader, "Чистый val (лабораторные условия)")

# ── Оценка на зашумлённом val (полевые условия) ──────────────────
val_ds_heavy = PlantVillageDataset(hf_train, val_idx, train_tf_heavy)
val_loader_heavy = DataLoader(val_ds_heavy, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)

# Несколько прогонов с разным шумом → усредняем (имитирует реальный разброс)
ROBUST_RUNS = 3
accs, f1s = [], []
for run in range(1, ROBUST_RUNS + 1):
    acc_r, f1_r, _, _ = evaluate(val_loader_heavy, f"Зашумлённый val — прогон {run}/{ROBUST_RUNS}")
    accs.append(acc_r); f1s.append(f1_r)

print(f"\n{'='*55}")
print(f"  ИТОГ  (среднее по {ROBUST_RUNS} прогонам с шумом)")
print(f"{'='*55}")
print(f"  Чистый val     : acc={acc_clean*100:.2f}%  F1={f1_clean:.4f}")
print(f"  Зашумлённый val: acc={np.mean(accs)*100:.2f}% ± {np.std(accs)*100:.2f}%  "
      f"F1={np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
print(f"  Деградация     : Δacc={( acc_clean - np.mean(accs))*100:.2f} п.п.")

## 7. Графики (Loss / Accuracy / F1)

In [ ]:
import re

# Восстанавливаем history и best_val_acc из лога если ячейка запущена отдельно
if 'history' not in dir() or not history["train_loss"]:
    log_path = os.path.join(SAVE_DIR, "train.log")
    history  = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "val_f1": []}
    best_val_acc = 0.0
    pattern = re.compile(
        r"^\s*(\d+)\s+\S+\s+([\d.]+)\s+([\d.]+)%\s+([\d.]+)\s+([\d.]+)%\s+([\d.]+)"
    )
    with open(log_path) as f:
        for line in f:
            m = pattern.match(line)
            if m:
                history["train_loss"].append(float(m.group(2)))
                history["train_acc"].append(float(m.group(3)) / 100)
                history["val_loss"].append(float(m.group(4)))
                history["val_acc"].append(float(m.group(5)) / 100)
                history["val_f1"].append(float(m.group(6)))
                best_val_acc = max(best_val_acc, float(m.group(5)) / 100)
    print(f"Загружено из лога: {len(history['train_loss'])} эпох, best_val_acc={best_val_acc*100:.2f}%")

epochs = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle(f"{MODEL_NAME} — PlantVillage (best val acc: {best_val_acc*100:.2f}%)", fontsize=13)

ax = axes[0]
ax.plot(epochs, history["train_loss"], label="train", marker=".")
ax.plot(epochs, history["val_loss"],   label="val",   marker=".")
ax.set_title("Cross-Entropy Loss")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(epochs, [v*100 for v in history["train_acc"]], label="train", marker=".")
ax.plot(epochs, [v*100 for v in history["val_acc"]],   label="val",   marker=".")
ax.set_title("Accuracy")
ax.set_xlabel("Epoch"); ax.set_ylabel("Acc (%)")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f"))
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[2]
ax.plot(epochs, history["val_f1"], label="val macro-F1", marker=".", color="green")
ax.set_title("Macro F1 (val)")
ax.set_xlabel("Epoch"); ax.set_ylabel("F1")
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(SAVE_DIR, "curves.png")
plt.savefig(plot_path, dpi=120)
plt.show()
print(f"Saved → {plot_path}")

## 8. Per-class accuracy

In [ ]:
# Восстанавливаем all_preds / all_labels если ячейка запущена отдельно
if 'all_preds' not in dir() or not isinstance(all_preds, np.ndarray):
    best_ckpt = os.path.join(SAVE_DIR, "best.pt")
    model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(DEVICE)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                preds = model(imgs).argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    print(f"Переоценка выполнена: {len(all_preds)} образцов")

per_class_acc = np.array([
    (all_preds[all_labels == c] == c).mean() if (all_labels == c).sum() > 0 else 0.0
    for c in range(NUM_CLASSES)
])

sorted_idx = np.argsort(per_class_acc)

fig, axes = plt.subplots(1, 2, figsize=(18, 5))
fig.suptitle(f"{MODEL_NAME} — Per-class Accuracy (PlantVillage)", fontsize=13)

ax = axes[0]
colors = ["#d62728" if per_class_acc[i] < 0.7 else
          "#ff7f0e" if per_class_acc[i] < 0.9 else
          "#2ca02c" for i in sorted_idx]
ax.bar(range(NUM_CLASSES), per_class_acc[sorted_idx] * 100, color=colors, width=0.8)
ax.set_title("Все классы (sorted)")
ax.set_xlabel("Класс (ранг)"); ax.set_ylabel("Accuracy (%)")
ax.axhline(per_class_acc.mean() * 100, color="navy", linestyle="--", linewidth=1.2,
           label=f"mean {per_class_acc.mean()*100:.1f}%")
ax.legend(); ax.grid(True, axis="y", alpha=0.3); ax.set_xticks([])

ax = axes[1]
n = 15
worst_n = sorted_idx[:n]
ax.barh(np.arange(n), per_class_acc[worst_n] * 100, color="#d62728", alpha=0.8)
ax.set_yticks(np.arange(n))
ax.set_yticklabels([CLASS_NAMES[i] for i in worst_n], fontsize=8)
ax.set_title(f"Топ-{n} худших классов")
ax.set_xlabel("Accuracy (%)")
ax.axvline(per_class_acc.mean() * 100, color="navy", linestyle="--", linewidth=1.2)
ax.grid(True, axis="x", alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(SAVE_DIR, "per_class_acc.png")
plt.savefig(plot_path, dpi=120)
plt.show()
print(f"Saved → {plot_path}")
print(f"Mean per-class acc : {per_class_acc.mean()*100:.2f}%")
print(f"Std  per-class acc : {per_class_acc.std()*100:.2f}%")
print(f"Min  per-class acc : {per_class_acc.min()*100:.2f}%  ({CLASS_NAMES[per_class_acc.argmin()]})")